In [2]:
import sire as sr
import BioSimSpace as BSS

import json
from pathlib import Path

In [3]:
def load_edges(path):
    data = json.loads(Path(path).read_text())
    return {
        e["id"]: {
            **e,
            "atom_mapping": {int(a): int(b) for a, b in e["atom_mapping"]},
        }
        for e in data["edges"]
    }

In [79]:
data = load_edges("campaign.json")

for i, edge in enumerate(data):
    print(f"Processing edge {i}: {edge}")



    # 1. Create an alchemical transformation
    edge_data = data[edge]

    # load associated input files
    ligand_a = BSS.IO.readMolecules([edge_data["ligand_a_files"]["coordinates"], edge_data["ligand_a_files"]["topology"]])[0]
    ligand_b = BSS.IO.readMolecules([edge_data["ligand_b_files"]["coordinates"], edge_data["ligand_b_files"]["topology"]])[0]

    # perform nAtoms assert to check that we are loading the same number of atoms as was recorded for the mapping
    assert ligand_a.nAtoms() == edge_data["n_atoms_a"]
    assert ligand_b.nAtoms() == edge_data["n_atoms_b"]

    is_ring_break = edge_data.get("is_ring_break")

    if is_ring_break is True:
        try:
            merged = BSS.Align.merge(
                ligand_a, ligand_b, edge_data["atom_mapping"],
                allow_ring_breaking=True,
                allow_ring_size_change=True,
            )
        except BSS._Exceptions.IncompatibleError:
            merged = BSS.Align.merge(
                ligand_a, ligand_b, edge_data["atom_mapping"],
                allow_ring_breaking=True,
                allow_ring_size_change=True,
                force=True,
        )
    elif is_ring_break is False:
        merged = BSS.Align.merge(ligand_a, ligand_b, edge_data["atom_mapping"])
    else:
        raise ValueError(
            f"edge did not declare whether it involves ring-breaking or not "
            f"(got {is_ring_break!r})"
    )

    # 2. Setup free leg
    box, angles = BSS.Box.truncatedOctahedron(10 * BSS.Units.Length.nanometer)


    box_min, box_max = merged.getAxisAlignedBoundingBox()
    box_size = [y - x for x, y in zip(box_min, box_max)]
    padding = 15 * BSS.Units.Length.angstrom
    box_length = max(box_size) + 2 * padding

    free_leg = BSS.Solvent.tip3p(molecule=merged, box=3 * [box_length], angles=angles, ion_conc=0.15, is_neutral=True)
    free_leg.reduceBoxVectors()
    BSS.Stream.save(free_leg, f"alchemical_somd2_run_files/{edge}_free_leg")


    # 3. Setup bound leg
    pcsk9 = BSS.IO.readMolecules(["../../data/proteins/7S5G_L_P_capped_pH_7_4_amber_version_relabelled.pdb"])
    pcsk9 = BSS.Parameters.ff14SB(pcsk9[0], ensure_compatible=False).getMolecule()
    pcsk9_holo = pcsk9 + merged

    box, angles = BSS.Box.truncatedOctahedron(10 * BSS.Units.Length.nanometer)
    box_min, box_max = pcsk9_holo.getAxisAlignedBoundingBox()
    box_size = [y - x for x, y in zip(box_min, box_max)]
    padding = 15 * BSS.Units.Length.angstrom
    box_length = max(box_size) + 2 * padding

    bound_leg = BSS.Solvent.tip3p(molecule=pcsk9_holo, box=3 * [box_length], angles=angles, ion_conc=0.15, is_neutral=True)
    bound_leg.reduceBoxVectors()
    BSS.Stream.save(bound_leg, f"alchemical_somd2_run_files/{edge}_bound_leg")


Processing edge 0: compound_17_rb_no_ole_to_5
Processing edge 1: compound_17_rb_only_to_17_rb_no_ole
Processing edge 2: compound_17_to_17_rb_only
Processing edge 3: compound_17_to_19
Processing edge 4: compound_17_to_22_unmap_ole_h
Processing edge 5: compound_17_to_25
Processing edge 6: compound_18_no_ole_to_17_unmap_ole_h
Processing edge 7: compound_18_redo_to_18_no_ole_unmap_ole_h
Processing edge 8: compound_22_to_18_unmap_ole_h
Processing edge 9: compound_25_to_29
Processing edge 10: compound_2_to_7_twisted
Processing edge 11: compound_35_to_25_corrected_rb_step1
Processing edge 12: compound_35_to_25_corrected_rb_step2
Processing edge 13: compound_5_no_ole_to_6_alt_conf_unmap_ole_h
Processing edge 14: compound_5_to_5_no_ole_unmap_ole_h
Processing edge 15: compound_7_rb_to_7_reconnected
Processing edge 16: compound_7_reconnected_to_5
Processing edge 17: compound_7_to_7_rb
